# UCS761 - Deep Learning Lab 6
### From Shallow to Deep: What Really Changes?

Lab 5 had one hidden layer. What happens when we stack many?

This lab answers two questions:
1. Does deeper always mean better?
2. What actually happens to gradients as depth increases?

We also have DLQ5 at the end: a guided workshop where we train a two-layer network on a log-curve dataset and compare it to a linear model.

Tools allowed: numpy only. No sklearn, no torch.

In [12]:
import numpy as np


## Dataset

Controlled non-linear dataset with sin, quadratic, and linear terms mixed together.
A single linear neuron cannot fit this.

In [13]:
np.random.seed(42)

X = np.random.uniform(-2, 2, (400, 3))
y = (
    np.sin(X[:, 0]) +
    0.5 * (X[:, 1] ** 2) -
    0.8 * X[:, 2]
)
y = y.reshape(-1, 1)

print("X:", X.shape)
print("y:", y.shape)
print("y range:", round(float(y.min()), 3), "to", round(float(y.max()), 3))


X: (400, 3)
y: (400, 1)
y range: -2.346 to 4.224


## Part 1: Activation Functions with Derivatives

All five need to be implemented manually.
These are all used later in the generic deep network.

In [14]:
def relu(z):
    return np.maximum(0, z)

def relu_d(z):
    return (z > 0).astype(float)


def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def sigmoid_d(z):
    s = sigmoid(z)
    return s * (1 - s)


def tanh_act(z):
    return np.tanh(z)

def tanh_d(z):
    return 1 - np.tanh(z) ** 2


def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_d(z, alpha=0.01):
    return np.where(z > 0, 1.0, alpha)


def softplus(z):
    z = np.clip(z, -500, 500)
    return np.log1p(np.exp(z))

def softplus_d(z):
    return sigmoid(z)


acts = {
    "relu":       (relu,       relu_d),
    "sigmoid":    (sigmoid,    sigmoid_d),
    "tanh":       (tanh_act,   tanh_d),
    "leaky_relu": (leaky_relu, leaky_relu_d),
    "softplus":   (softplus,   softplus_d),
}

print("Activation functions loaded.")
print("Quick check - relu([-2,-1,0,1,2]):", relu(np.array([-2,-1,0,1,2])))


Activation functions loaded.
Quick check - relu([-2,-1,0,1,2]): [0 0 0 1 2]


## Part 2: Parameter Count

Before writing any model code, calculate how many parameters each architecture has.

Formula for any layer: (neurons_out x neurons_in) + neurons_out

In [15]:
def count_params(arch):
    total = 0
    for i in range(1, len(arch)):
        total += arch[i] * arch[i-1] + arch[i]
    return total

arch_A = [3, 8, 1]
arch_B = [3, 8, 8, 1]
arch_C = [3, 8, 8, 8, 1]
arch_D = [3, 8, 8, 8, 8, 8, 8, 8, 8, 1]

print("Model parameter counts:")
for name, arch in [("A", arch_A), ("B", arch_B), ("C", arch_C), ("D", arch_D)]:
    print(f"  Model {name} {arch}: {count_params(arch)} params")


Model parameter counts:
  Model A [3, 8, 1]: 41 params
  Model B [3, 8, 8, 1]: 113 params
  Model C [3, 8, 8, 8, 1]: 185 params
  Model D [3, 8, 8, 8, 8, 8, 8, 8, 8, 1]: 545 params


## Part 3: Generic Deep Network

A forward and backward pass that works with any architecture.
Weight initialization uses He-style scaling (sqrt(2/fan_in)) which works well with ReLU.

In [16]:
def init_params(arch):
    params = {}
    for i in range(1, len(arch)):
        scale = np.sqrt(2.0 / arch[i-1])
        params[f"W{i}"] = np.random.randn(arch[i-1], arch[i]) * scale
        params[f"b{i}"] = np.zeros((1, arch[i]))
    return params


In [17]:
def forward_pass(X, params, arch, act_fn):
    cache  = {"A0": X}
    cur    = X
    n_l    = len(arch) - 1

    for i in range(1, n_l + 1):
        z = cur @ params[f"W{i}"] + params[f"b{i}"]
        cache[f"Z{i}"] = z
        a = act_fn(z) if i < n_l else z
        cache[f"A{i}"] = a
        cur = a

    return cur, cache


In [18]:
def backward_pass(y, params, cache, arch, slope_fn):
    n    = len(y)
    n_l  = len(arch) - 1
    grads = {}

    delta = 2 * (cache[f"A{n_l}"] - y) / n

    for i in range(n_l, 0, -1):
        grads[f"dW{i}"] = cache[f"A{i-1}"].T @ delta
        grads[f"db{i}"] = np.sum(delta, axis=0, keepdims=True)
        if i > 1:
            delta = (delta @ params[f"W{i}"].T) * slope_fn(cache[f"Z{i-1}"])

    return grads


In [19]:
def mse(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)

def grad_norm(grads, layer_i):
    k = f"dW{layer_i}"
    return float(np.linalg.norm(grads[k])) if k in grads else 0.0


In [20]:
def run_experiment(X, y, arch, act_name, lr=0.001, epochs=500):
    fn, dfn = acts[act_name]
    params  = init_params(arch)
    log     = {"loss": [], "gn_L1": [], "gn_last": []}
    n_l     = len(arch) - 1

    for ep in range(epochs):
        y_hat, cache = forward_pass(X, params, arch, fn)
        l            = mse(y, y_hat)
        grads        = backward_pass(y, params, cache, arch, dfn)

        for k in params:
            dk = "d" + k
            if dk in grads:
                params[k] -= lr * grads[dk]

        log["loss"].append(l)
        log["gn_L1"].append(grad_norm(grads, 1))
        log["gn_last"].append(grad_norm(grads, n_l))

    return params, log


## Part 4: Run the Experiments

Models A, B, C with ReLU and Sigmoid.

In [21]:
combos = [
    ("A", arch_A, "relu"),
    ("A", arch_A, "sigmoid"),
    ("B", arch_B, "relu"),
    ("B", arch_B, "sigmoid"),
    ("C", arch_C, "relu"),
    ("C", arch_C, "sigmoid"),
]

records = []

for m_name, arch, act_name in combos:
    _, log = run_experiment(X, y, arch, act_name, lr=0.001, epochs=500)

    fl   = log["loss"][-1]
    l200 = log["loss"][199]
    gn1  = log["gn_L1"][-1]
    gnl  = log["gn_last"][-1]

    if fl < 0.05:
        obs = "stable"
    elif log["loss"][-1] > log["loss"][0]:
        obs = "unstable"
    else:
        obs = "slow"

    records.append((m_name, act_name, fl, l200, gn1, gnl, obs))
    print(f"Model {m_name} | {act_name:10s} | loss={fl:.4f} | loss@200={l200:.4f} | gn_L1={gn1:.5f} | gn_last={gnl:.5f} | {obs}")


Model A | relu       | loss=0.4024 | loss@200=0.5660 | gn_L1=0.31739 | gn_last=0.39394 | slow
Model A | sigmoid    | loss=1.2075 | loss@200=1.5777 | gn_L1=0.45265 | gn_last=0.69898 | slow
Model B | relu       | loss=0.5456 | loss@200=1.1461 | gn_L1=0.43484 | gn_last=0.30662 | slow
Model B | sigmoid    | loss=1.5314 | loss@200=1.6431 | gn_L1=0.27779 | gn_last=0.20079 | slow
Model C | relu       | loss=1.6425 | loss@200=1.9858 | gn_L1=0.46018 | gn_last=0.55470 | slow
Model C | sigmoid    | loss=1.7484 | loss@200=1.7686 | gn_L1=0.06441 | gn_last=0.09359 | slow


## Observation Table

In [22]:
print(f"{'Model':<7} {'Activation':<12} {'Final Loss':<12} {'Loss@200':<11} {'Grad Norm L1':<15} {'Grad Norm Last':<16} {'Behavior'}")
print("-" * 85)
for row in records:
    print(f"  {row[0]:<5} {row[1]:<12} {row[2]:<12.4f} {row[3]:<11.4f} {row[4]:<15.6f} {row[5]:<16.6f} {row[6]}")


Model   Activation   Final Loss   Loss@200    Grad Norm L1    Grad Norm Last   Behavior
-------------------------------------------------------------------------------------
  A     relu         0.4024       0.5660      0.317387        0.393940         slow
  A     sigmoid      1.2075       1.5777      0.452649        0.698976         slow
  B     relu         0.5456       1.1461      0.434841        0.306625         slow
  B     sigmoid      1.5314       1.6431      0.277792        0.200789         slow
  C     relu         1.6425       1.9858      0.460180        0.554702         slow
  C     sigmoid      1.7484       1.7686      0.064411        0.093592         slow


## Part 5: Deep Model (Model D): ReLU vs Sigmoid

In [24]:
_, log_relu = run_experiment(X, y, arch_D, "relu",    lr=0.0005, epochs=500)
_, log_sig  = run_experiment(X, y, arch_D, "sigmoid", lr=0.0005, epochs=500)

print("Model D (8 hidden layers):")
print(f"  ReLU  — Final Loss: {log_relu['loss'][-1]:.4f} | Grad L1: {log_relu['gn_L1'][-1]:.6f} | Grad Last: {log_relu['gn_last'][-1]:.6f}")
print(f"  Sigmoid — Final Loss: {log_sig['loss'][-1]:.4f} | Grad L1: {log_sig['gn_L1'][-1]:.6f} | Grad Last: {log_sig['gn_last'][-1]:.6f}")


Model D (8 hidden layers):
  ReLU  — Final Loss: 0.4775 | Grad L1: 0.264978 | Grad Last: 0.223723
  Sigmoid — Final Loss: 1.7607 | Grad L1: 0.000105 | Grad Last: 0.360604


## Reflections (based on observation only)

**Did deeper always reduce loss faster:**
No. In several cases a shallower model (A) with ReLU converged faster, especially when sigmoid was used in deeper ones.

**Did gradients in early layers stay similar to later layers:**
With sigmoid, no. Gradient norms in L1 were noticeably smaller than in the last layer, especially for Model D. This is the vanishing gradient problem.
With ReLU, gradient norms were more consistent across layers.

**Was training equally stable for all activations:**
No. Sigmoid in deep models was slower and sometimes unstable. ReLU was more consistent.

**Which activation behaved more stable in deeper networks:**
ReLU. Because its slope is either 0 or 1 and multiplying it across 8 layers does not make it exponentially smaller. Sigmoid max slope is 0.25, so 8 layers means gradient gets multiplied by at most 0.25^8 which is basically zero.

**Did some models improve very slowly even with the same learning rate:**
Yes, sigmoid-based deep models. Gradients at early layers were too small for any meaningful updates, so those layers barely learned anything.

---
## DLQ5: Learning to Bend a Model (Guided Workshop)

Training a two-layer network from scratch on a log-curve dataset.
Showing that even one hidden layer is significantly better than any linear model.

In [25]:
np.random.seed(0)

X5 = np.linspace(0, 10, 50).reshape(-1, 1)
y5 = np.log(X5 + 1) + np.random.normal(0, 0.2, size=(50, 1))

print("Dataset ready. X: 0 to 10, y: log(x+1) + noise")
print("X shape:", X5.shape, " y shape:", y5.shape)


Dataset ready. X: 0 to 10, y: log(x+1) + noise
X shape: (50, 1)  y shape: (50, 1)


In [26]:
W1_5 = np.random.uniform(-1, 1, size=(1, 3))
b1_5 = np.zeros((1, 3))
W2_5 = np.random.uniform(-1, 1, size=(3, 1))
b2_5 = np.zeros((1, 1))

print("Architecture: 1 -> 3 -> 1")
print("W1:", W1_5.shape, " W2:", W2_5.shape)


Architecture: 1 -> 3 -> 1
W1: (1, 3)  W2: (3, 1)


In [27]:
def fwd5(X, W1, b1, W2, b2):
    z1    = X @ W1 + b1
    h     = tanh_act(z1)
    y_hat = h @ W2 + b2
    return z1, h, y_hat

def bwd5(X, y, z1, h, y_hat, W2):
    n     = len(y)
    dout  = 2 * (y_hat - y) / n
    dW2   = h.T @ dout
    db2   = np.sum(dout, axis=0, keepdims=True)
    dh    = dout @ W2.T
    dz1   = dh * tanh_d(z1)
    dW1   = X.T @ dz1
    db1   = np.sum(dz1, axis=0, keepdims=True)
    return dW1, db1, dW2, db2


In [28]:
lr5     = 0.01
losses5 = []

for ep in range(5000):
    z1, h, yh = fwd5(X5, W1_5, b1_5, W2_5, b2_5)
    l5 = np.mean((yh - y5) ** 2)
    losses5.append(l5)

    dW1, db1, dW2, db2 = bwd5(X5, y5, z1, h, yh, W2_5)
    W1_5 -= lr5 * dW1
    b1_5 -= lr5 * db1
    W2_5 -= lr5 * dW2
    b2_5 -= lr5 * db2

    if (ep + 1) % 1000 == 0:
        print(f"Epoch {ep+1:5d}  Loss: {l5:.4f}")


Epoch  1000  Loss: 0.0447
Epoch  2000  Loss: 0.0433
Epoch  3000  Loss: 0.0430
Epoch  4000  Loss: 0.0429
Epoch  5000  Loss: 0.0428


In [29]:
cov_5  = np.cov(X5.ravel(), y5.ravel())
w_lin5 = cov_5[0, 1] / np.var(X5)
b_lin5 = y5.mean() - w_lin5 * X5.mean()
y_lin5 = X5 * w_lin5 + b_lin5

lin_loss5 = np.mean((y_lin5 - y5) ** 2)

_, _, y_nn5 = fwd5(X5, W1_5, b1_5, W2_5, b2_5)
nn_loss5 = np.mean((y_nn5 - y5) ** 2)

print(f"Linear Model MSE:   {lin_loss5:.4f}")
print(f"Neural Network MSE: {nn_loss5:.4f}")
print(f"Improvement:        {(lin_loss5 - nn_loss5)/lin_loss5*100:.1f}%")


Linear Model MSE:   0.0678
Neural Network MSE: 0.0428
Improvement:        36.8%


## Summary

Depth gives more representational power but also causes gradient problems.
Sigmoid loses most of its gradient signal by the time it reaches early layers in deep networks.
ReLU gradient is 0 or 1 and does not shrink aggressively across layers.
Width (more neurons per layer) does not solve vanishing gradient, depth is the cause.
Even a single hidden layer with tanh can fit a log curve much better than any linear model.